# Ray and Ray Serve fundamentals

© 2026, Anyscale. All Rights Reserved

<div class="alert alert-block alert-info">
<b>Roadmap for this notebook</b>
<ol>
    <li>A brief Ray primer: tasks and actors.</li>
    <li>Why not just use a Ray Core actor as a service?</li>
    <li>Your first Ray Serve deployment.</li>
    <li>Request routing and admission control.</li>
    <li>HTTP ingress with FastAPI and Ray Serve.</li>
    <li>Multiple Deployment Composition with Ray Serve.</li>
    <li>Ray Serve vs plain Kubernetes Deployments and Services.</li>
</ol>
</div>

**Imports**

In [ ]:
import asyncio
import time

import numpy as np
import requests
from fastapi import FastAPI
from pydantic import BaseModel
from ray import serve
from ray.serve.handle import DeploymentHandle
import ray

## 1. A brief Ray primer

Ray Serve is a library built on Ray, so a little Ray vocabulary pays off for the whole notebook.

Start (or connect to) Ray.

In [ ]:
ray.init(ignore_reinit_error=True)

The same call boots a one-process cluster on a laptop or attaches to a multi-node cluster, so everything below is identical in both places.

Whichever it is, the shape is the same, and the worker processes in it are where Serve later places its replica actors.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/ray_cluster_architecture.png" loading="lazy" width="1000">

### 1.1 Tasks

Decorating a python function with `@ray.remote` turns it into a remote function

In [ ]:
@ray.remote
def expensive_square(x: int) -> int:
    time.sleep(2)
    return x * x

expensive_square

You call the remote function with `.remote()`

In [ ]:
expensive_square.remote(2)

Ray immediately schedules a **Ray task** on the cluster and hands back an `ObjectRef`.

Each `.remote()` call becomes one task, and the cluster places it on whichever worker is free:

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/python_to_ray_task_map.png" loading="lazy" width="900">

`ray.get()` is what blocks until the value is ready.

In [ ]:
%%time
ray.get([expensive_square.remote(i) for i in range(4)])

This fires four tasks at once, runs them in parallel, and they finish in about the time of a single task, not the sum of all four.

### 1.2 Actors

A task is **stateless** - i.e. it does not leave any application-level state on the worker process that it ran on. 

An **actor** on the other hand is a long-lived worker process that keeps state between calls. 

Decorating a class with `ray.remote` makes it an ActorClass.

In [ ]:
@ray.remote
class Counter:
    def __init__(self) -> None:
        self.n = 0
    def incr(self) -> None:
        self.n += 1
    def get(self) -> int:
        return self.n

Counter

You instantiate the actor class with `.remote`. Ray will immediately try to schedule the actor and returns a **handle** to communicate with the actor process.

In [ ]:
counter_handle = Counter.remote()

You use the counter_handle to invoke a method on the actor process using .remote().

In [ ]:
counter_handle.get.remote()

This will immediately schedule an actor task and return an `ObjectRef`

Creating an actor is not a local call: the global control store leases a worker for it, which is why the handle came back before `__init__` finished.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/actor_creation.png" loading="lazy" width="1000">

You can then use `ray.get` to queue multiple actor tasks and wait to retrieve their value.

In [ ]:
for _ in range(3):
    counter_handle.incr.remote() 
ray.get(counter_handle.get.remote())

## 2. Why not just use a Ray Core actor as a service?

You already have actors, so why a serving framework at all? Try the naive version: hold the model in a raw actor.

First, the pure scoring logic. Keep it free of Ray so it stays testable on its own.

In [ ]:
class RankerLogic:
    def __init__(self, dim: int = 16) -> None:
        self.w = np.ones(dim, dtype="float32")   # stand-in for learned weights
    def score(self, user: list[float], item: list[float]) -> float:
        return float(self.w @ (np.asarray(user) * np.asarray(item)))

Score a `(user, item)` pair with a plain in-process call.

In [ ]:
logic = RankerLogic()
logic.score([1.0] * 16, [0.5] * 16)

Note: keeping `RankerLogic` pure, separate from any serving wrapper, is the pattern to follow. The logic is unit-testable with no cluster.

Now wrap it in a Ray actor to make it a "service."

In [ ]:
@ray.remote
class RawRankerActor:
    def __init__(self) -> None:
        self.model = RankerLogic()       # load weights once, in the actor process
    def score(self, user: list[float], item: list[float]) -> float:
        return self.model.score(user, item)

`RawRankerActor.remote()` gives you an actor handle, and you score by calling a method on it.

In [ ]:
actor = RawRankerActor.remote()
ray.get(actor.score.remote([1.0] * 16, [0.5] * 16))

This works for a single in-process caller, but it is not a service.

A real online service needs a whole layer around the model that a raw actor does not give you. That layer is Ray Serve, itself built from actors, and the rest of this notebook adds each piece in turn.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/serve_gap.png" loading="lazy" width="980">

## 3. Your first Ray Serve deployment

A deployment turns your class into a managed group of replica actors, reached through a handle rather than by holding an actor reference yourself.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/deployment_lifecycle.png" loading="lazy" width="760">

`@serve.deployment` turns a class into a managed deployment. Serve runs N replica actors for it, and `__init__` runs once per replica, which is where you load weights.

In [ ]:
@serve.deployment(num_replicas=2, ray_actor_options={"num_cpus": 1})
class Ranker:
    def __init__(self, dim: int = 16) -> None:
        self.model = RankerLogic(dim)         # runs ONCE per replica (weights load here)
    async def __call__(self, user: list[float], item: list[float]) -> float:
        return self.model.score(user, item)

`Ranker.bind()` builds a blueprint. Nothing runs yet.

In [ ]:
blueprint = Ranker.bind()
blueprint

`serve.run` materializes the blueprint into live replicas and returns a `DeploymentHandle`.

In [ ]:
handle = serve.run(blueprint, name="ranker")
handle

Call the handle like a function. It returns a `DeploymentResponse` immediately and does not block.

In [ ]:
response = handle.remote([1.0] * 16, [0.5] * 16)
response

`await` is what waits for the value.

In [ ]:
print(await response)

Tear down before the next section starts a fresh app.

In [ ]:
serve.shutdown()

## 4. Request routing and admission control

A handle fronting two replicas has to pick one per request. Two separate decisions govern that, and this section takes them in turn: **routing** picks which replica gets the request, and **admission** decides whether the request is taken at all.

### 4.1 Load-aware routing: the second request finds the idle replica

The router lives on every `DeploymentHandle` and samples two replicas rather than scanning all of them.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/p2c_routing.png" loading="lazy" width="900">

The default policy is `PowerOfTwoChoicesRequestRouter`: pick two replicas at random (with exactly two replicas, both are always sampled), ask each for its queue length and whether it can accept, and route to the lower-queue accepter. If neither accepts it retries with backoff.

- **It lives on the handle, so it covers every hop.** Even a one-deployment app routes: the Python proxy holds a handle for ingress, and each replica holds one for downstream calls.
- **It is pluggable.** Swap the policy with `RequestRouterConfig(request_router_class=...)`.

To watch routing choose, make one replica busy and see where the next request lands. Give a 2-replica deployment a `__call__` that sleeps, so a request can be held in flight while we send another.

In [ ]:
@serve.deployment(num_replicas=2)
class Sleeper:
    async def __call__(self, t: float) -> str:
        await asyncio.sleep(t)
        return serve.get_replica_context().replica_id.unique_id

h = serve.run(Sleeper.bind(), name="ranker")

`__call__` returns the id of whichever replica served it. Fire a slow request to occupy one replica, pause so the router observes that replica's queue, then send a fast one and see where it lands.

In [ ]:
slow = h.remote(5.0)            # occupies one replica, stays in flight
await asyncio.sleep(0.5)        # let the router observe the busy replica
fast_id = await h.remote(0.1)   # routed while the slow one is still running
slow_id = await slow
print(fast_id, slow_id)
assert fast_id != slow_id

The fast request skipped the replica already busy with the slow one. Power of Two compared in-flight counts, 1 against 0, and preferred the idle replica. No admission limit was in play: with the default `max_ongoing_requests=5` the busy replica could still have accepted the request; the router simply prefers the less-loaded one.

In [ ]:
serve.shutdown()

### 4.2 The admission knobs: `max_ongoing_requests` and `max_queued_requests`

Routing chooses among replicas that can take work; admission decides whether the request is taken at all. Two bounds govern it, one on each side of the handle: a replica accepts only while its in-flight count is below `max_ongoing_requests`, and requests with nowhere to go wait in a per-handle queue bounded by `max_queued_requests`, then are rejected with `BackPressureError`.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/admission_control.png" loading="lazy" width="900">

Reproduce that capacity: 2 replicas at `max_ongoing_requests=1` and a queue of 4. Admission is decided before a replica is chosen, so a *simultaneous* burst against empty slots would reject at the queue bound alone. Pre-filling the slots is what makes the full 2 + 4 capacity show up.

In [ ]:
@serve.deployment(num_replicas=2, max_ongoing_requests=1, max_queued_requests=4)
class Sleeper:
    async def __call__(self, t: float) -> str:
        await asyncio.sleep(t)
        return serve.get_replica_context().replica_id.unique_id

h = serve.run(Sleeper.bind(), name="ranker")

Fire one request per replica to occupy both slots, then pause so the router assigns them before the burst.

In [ ]:
busy = [h.remote(5.0) for _ in range(2)]
await asyncio.sleep(1.0)

Now fire a burst of six against the full slots: four fit the bounded queue, the last two have nowhere to go.

In [ ]:
burst = [h.remote(5.0) for _ in range(6)]
results = await asyncio.gather(*burst, return_exceptions=True)
print([type(x).__name__ for x in results])

Four are admitted (queued, then served) and two come back as `BackPressureError`. Drain the two occupiers to finish.

In [ ]:
await asyncio.gather(*busy)

Note: `max_ongoing_requests` defaults to **5** and is enforced at the replica by a semaphore; it is the accept/reject half of the handshake. `max_queued_requests` defaults to **-1** (unbounded); the queue lives on the caller side, per handle, and overflow raises `BackPressureError`, surfaced as **HTTP 503** at the proxy.

<div class="alert alert-block alert-warning">
With the default <code>max_queued_requests=-1</code> the queue is unbounded, so backpressure never rejects and you rely on timeouts or cancellation instead. Set a finite <code>max_queued_requests</code> to shed load fast.
</div>

In [ ]:
serve.shutdown()

## 5. HTTP ingress with FastAPI and Ray Serve

Real clients speak HTTP, not Python handle calls. The proxy terminates HTTP and the FastAPI app runs inside the replica.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/http_ingress.png" loading="lazy" width="900">

`@serve.ingress(app)` mounts a FastAPI app on the deployment, and a Pydantic model gives the request body a typed contract.

In [ ]:
fastapi_app = FastAPI()

class ScoreRequest(BaseModel):
    user: list[float]
    item: list[float]

@serve.deployment
@serve.ingress(fastapi_app)
class RankerAPI:
    def __init__(self) -> None:
        self.model = RankerLogic()
    @fastapi_app.post("/score")
    def score(self, req: ScoreRequest) -> dict:
        return {"score": self.model.score(req.user, req.item)}

serve.run(RankerAPI.bind(), name="ranker")

The proxy now listens on `localhost:8000`. Query it with an ordinary HTTP client.

In [ ]:
r = requests.post("http://127.0.0.1:8000/score",
                  json={"user": [1.0] * 16, "item": [0.5] * 16})
print(r.status_code, r.json())   # 200 {'score': 8.0}

<div class="alert alert-block alert-info">
The HTTP proxy listens on <code>:8000</code> and forwards <code>/score</code> to the replica. The FastAPI app runs <i>inside</i> the replica, so Pydantic validates the body there and returns <b>422</b> on a bad shape; the proxy only matches the route and relays the response.
</div>

A malformed body proves the contract: send a string where a list of floats is expected.

In [ ]:
bad = requests.post("http://127.0.0.1:8000/score",
                    json={"user": "not-a-vector", "item": [0.5] * 16})
print(bad.status_code)

Pydantic rejected it with **422** before your code ran.

In [ ]:
serve.shutdown()

## 6. Multiple Deployment Composition with Ray Serve

Online inference is usually a graph, not a single model. Each stage is its own deployment, and a parent deployment calls them in turn. Read this as the general shape of any multi-stage online inference pipeline.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/composition_graph.png" loading="lazy" width="900">

Here a `FeaturePrep` stage normalizes the raw vectors before the `Ranker` scores them.

In [ ]:
@serve.deployment
class FeaturePrep:
    async def __call__(self, raw_user: list[float], raw_item: list[float]) -> tuple[list[float], list[float]]:
        norm = lambda v: (np.asarray(v) / (np.linalg.norm(v) + 1e-9)).tolist()
        return norm(raw_user), norm(raw_item)   # unit-normalize both vectors

A parent deployment receives the downstream handles at bind time and calls them with `await handle.remote(...)`, one hop per stage.

In [ ]:
@serve.deployment
class Pipeline:
    def __init__(self, prep: DeploymentHandle, ranker: DeploymentHandle) -> None:
        self.prep, self.ranker = prep, ranker
    async def __call__(self, raw_user: list[float], raw_item: list[float]) -> float:
        user, item = await self.prep.remote(raw_user, raw_item)   # hop 1: featurize
        return await self.ranker.remote(user, item)               # hop 2: score

`.bind()` wires the graph: `FeaturePrep` and `Ranker` are bound into `Pipeline`, then the whole graph deploys with one `serve.run`.

In [ ]:
app = Pipeline.bind(FeaturePrep.bind(), Ranker.bind())
handle = serve.run(app, name="ranker")

Call the graph through its handle, exactly like the single deployment before.

In [ ]:
print(await handle.remote([3.0] * 16, [4.0] * 16))

The result is ~1.0: the vectors are unit-normalized by `FeaturePrep`, then scored by `Ranker`.

<div class="alert alert-block alert-info">
<b>Split vs fuse.</b> <b>Split</b> stages that scale differently or want different hardware (a CPU featurizer vs a GPU model) so each autoscales on its own. <b>Fuse</b> tiny always-together stages, since an extra hop is latency you do not need. Each <code>.remote()</code> is queue-aware with automatic backpressure.
</div>

In [ ]:
serve.shutdown()

## 7. Ray Serve vs plain Kubernetes Deployments and Services

A common question is why not just run the model behind a Kubernetes `Service`.

A `Service` gives you L4 load-balancing (round-robin, blind to per-pod in-flight count) and nothing else about serving. Everything that makes online inference good you assemble and operate yourself. Ray Serve gives those out of the box, in ordinary Python:

- **Load-aware routing.** Queue-aware Power of Two Choices on every handle (section 4), versus an L4 `Service` plus a separate L7 router and per-pod metrics you run yourself.
- **Dynamic batching.** A `@serve.batch` decorator that coalesces concurrent calls into one batched model call, versus a request-batching layer you build and tune.
- **Composition over efficient RPC.** An in-process call between deployments, with large payloads passed by reference through the object store (section 6), versus an HTTP/JSON graph that copies the full payload at every edge and leaves retries and backpressure to you.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/serve_vs_k8s_rpc_rev4.png" loading="lazy" width="1000">

### 7.1 Efficient RPC and the object store

Why is Serve composition cheaper than an HTTP graph? A `handle.remote(...)` between deployments is an ordinary Ray actor call, so two things hold:

- **gRPC transport.** Actor calls travel over gRPC with a binary payload, not HTTP/JSON, so there is no per-hop re-serialization an HTTP graph pays at every edge.
- **Large objects by reference.** Small args go inline; a large payload (tensor, image, embedding) is put once into the shared **object store** and passed as a reference. Co-located actors read it from shared memory with no copy, so one object feeds several downstream actors without being re-sent.
- **A parent forwards the reference**, never materializing the value just to re-send it.

<div class="alert alert-block alert-info">
For small, latency-sensitive payloads you can send the value directly over gRPC instead of through the object store: set <code>RAY_SERVE_USE_GRPC_BY_DEFAULT=1</code>. The default by-reference path wins for large payloads. Experimental, not a stable public API.
</div>

### 7.2 How Serve runs on Kubernetes when you want it: KubeRay RayService

If your platform standardizes on Kubernetes, you do not give any of this up. KubeRay's `RayService` runs your Serve apps on a Ray cluster inside k8s, so you keep queue-aware routing, Python composition, and object-store RPC, and get k8s-native deploys and upgrades on top.

```yaml
apiVersion: ray.io/v1
kind: RayService
spec:
  serveConfigV2: |              # your Serve apps and deployments
    applications: [ ... ]
  rayClusterConfig: { ... }     # the Ray cluster Serve runs on
  # Upgrades: blue/green (NewCluster) OR incremental Gateway-API traffic shift
  #           (NewClusterWithIncrementalUpgrade, alpha, KubeRay v1.5+)
```

<div class="alert alert-block alert-info">
<b>Reference implementation:</b> the runnable Ranker lives in <code>code/classic/ranking/ranker.py</code> (the same <code>RankerLogic</code> vs <code>Ranker</code> split shown here, scaled up to a real two-tower model) with its paired <code>service.yaml</code>.
</div>